# Learned matching

Models trained on the train labels; precision, recall and F1 on the validation labels.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
PAIRS = ["wikidata-ror", "deqar-ror"]

In [ ]:
matching = pd.read_csv("results/matching.csv")
matching[["round", "pair", "matcher", "threshold", "P", "R", "F1", "corr", "seconds"]]

In [ ]:
def plot_matching(df, rnd):
    fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * df["matcher"].nunique() + 1.5), sharey=True)
    for ax, pair in zip(axes, PAIRS):
        part = df[(df["round"] == rnd) & (df["pair"] == pair)].set_index("matcher")[["P", "R", "F1"]]
        part.plot.barh(ax=ax, width=0.8)
        ax.set_xlim(0, 1)
        ax.set_title(f"round {rnd}: {pair}")
        ax.legend(loc="lower right")
    plt.tight_layout()

## Round 2

In [ ]:
plot_matching(matching, 2)

## Round 3: translated names, and the extra feature part_word

In [ ]:
plot_matching(matching, 3)

## Rule-based against learned (best of each, round 2)

In [ ]:
rules = pd.read_csv("../03_rule_based/results/matching.csv")
best = []
for name, df in [("rule-based", rules), ("learned", matching)]:
    part = df[df["round"] == 2]
    best.append(part.loc[part.groupby("pair")["F1"].idxmax()].assign(method=name))
best = pd.concat(best)
best.pivot(index="pair", columns="method", values="F1").plot.bar(rot=0, ylim=(0.8, 1), figsize=(7, 4))
plt.ylabel("F1 (validation)")
best[["method", "pair", "matcher", "P", "R", "F1"]]